Assignment 1

f(x,y)=(x-3)^+2+(y+2)^2


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x,y):
    return (x-3)**2 + (y+2)**2

def gradient(x, y):
    df_dx = 2*(x-3)
    df_dy = 2*(y+2)
    return df_dx, df_dy

def gradient_descent_x(starting_x, learning_rate, iterations, y_fixed):
    x = starting_x
    path_x = [x]
    for i in range(iterations):
        df_dx, _ = gradient(x, y_fixed)
        x -= learning_rate * df_dx
        path_x.append(x)
        print(f"Iteration {i+1} (x): x = {x:.4f}, f(x, y) = {f(x, y_fixed):.4f}")
    return x, path_x

def gradient_descent_y(starting_y, learning_rate, iterations, x_fixed):
    y = starting_y
    path_y = [y]
    for i in range(iterations):
        _, df_dy = gradient(x_fixed, y)
        y -= learning_rate * df_dy
        path_y.append(y)
        print(f"Iteration {i+1} (y): y = {y:.4f}, f(x, y) = {f(x_fixed, y):.4f}")
    return y, path_y

starting_point = (0, 0)
learning_rate = 0.1
iterations = 10

print("Gradient Descent for X:")
minimum_x, path_x = gradient_descent_x(starting_point[0], learning_rate, iterations, starting_point[1])
print(f"\nMinimum x = {minimum_x:.4f}, f(x, y) = {f(minimum_x, starting_point[1]):.4f}")

print("\n\nGradient Descent for Y:")
minimum_y, path_y = gradient_descent_y(starting_point[1], learning_rate, iterations, starting_point[0])
print(f"\nMinimum y = {minimum_y:.4f}, f(x, y) = {f(starting_point[0], minimum_y):.4f}")

# Create side-by-side plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot for X
x_vals = np.linspace(-10, 10, 100)
Z_x = [(x - 3)**2 + (starting_point[1] + 2)**2 for x in x_vals]
ax1.plot(x_vals, Z_x, label="f(x, y_fixed)")
ax1.plot(path_x, [(x - 3)**2 + (starting_point[1] + 2)**2 for x in path_x], marker="o", color="red", label="Gradient Descent Path")
ax1.scatter(minimum_x, f(minimum_x, starting_point[1]), color="blue", label="Minimum")
ax1.set_xlabel("x")
ax1.set_ylabel("f(x, y)")
ax1.set_title("Gradient Descent for X")
ax1.legend()
ax1.grid()

# Plot for Y
y_vals = np.linspace(-10, 10, 100)
Z_y = [(starting_point[0] - 3)**2 + (y + 2)**2 for y in y_vals]
ax2.plot(y_vals, Z_y, label="f(x_fixed, y)")
ax2.plot(path_y, [(starting_point[0] - 3)**2 + (y + 2)**2 for y in path_y], marker="o", color="red", label="Gradient Descent Path")
ax2.scatter(minimum_y, f(starting_point[0], minimum_y), color="blue", label="Minimum")
ax2.set_xlabel("y")
ax2.set_ylabel("f(x, y)")
ax2.set_title("Gradient Descent for Y")
ax2.legend()
ax2.grid()

plt.tight_layout()
plt.show()

: 

Question 2: Deep Learning for Image Classification



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])  

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transforms)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transforms)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)
classes = trainset.classes


class CNN(nn.Module):
    def __init__(self, num_layers=4, num_classes=10):
        super(CNN, self).__init__()
        
        self.num_layers = num_layers

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), 
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        if self.num_layers == 8:
            self.block3 = nn.Sequential(
                nn.Conv2d(128, 256, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(256, 256, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )
            self.block4 = nn.Sequential(
                nn.Conv2d(256, 512, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv2d(512, 512, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(kernel_size=2, stride=2)
            )
            self.fc1 = nn.Linear(512*2*2, 512)
        else:  
            self.fc1 = nn.Linear(128*8*8, 512)
            
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        if self.num_layers == 8:
            x = self.block3(x)
            x = self.block4(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x) 

        return x
    
model4 = CNN(num_layers=4, num_classes=100).to(device)
model8 = CNN(num_layers=8, num_classes=100).to(device)
print(model4, model8)


criterion = nn.CrossEntropyLoss()
optimizer4 = optim.Adam(model4.parameters(), lr=0.001)
optimizer8 = optim.Adam(model8.parameters(), lr=0.001)

train_loader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

epochs = 10

for epoch in range(epochs):
    runningloss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer4.zero_grad()
        optimizer8.zero_grad()
        outputs4 = model4(inputs.to(device))
        outputs8 = model8(inputs.to(device))
        loss4 = criterion(outputs4, labels.to(device))
        loss8 = criterion(outputs8, labels.to(device))
        loss4.backward()
        loss8.backward()
        optimizer4.step()
        optimizer8.step()
        runningloss += loss4.item() + loss8.item()

    print(f'Epoch {epoch+1}, Loss: {runningloss/len(trainloader):.4f}')